# Selección de targets MOP con cobertura Rubin DP2

Ejecutar el notebook en orden. Genera tablas de visibilidad, parámetros de microlensing de MOP, métricas de la colección elegida, mapas del cielo y un reporte por target con coadds disponibles.


## 1. Configuración y conexiones

Elegí `DATA_RELEASE_NAME` entre `DP0.1`, `DP0.2`, `DP1` y `DP2`. El perfil configura servicio RSP, repositorio/collections Butler, tabla y columnas TAP, y datasets de coadd; `DP2` conserva exactamente la corrida actual.


In [8]:
from importlib import reload
from pathlib import Path

import pandas as pd
from lsst.daf.butler import Butler
from lsst.rsp import RSPDiscovery
import mop_api.client as mop_client
import target_selection_pipeline as pipeline
import data_release_config as release_config

# Recarga cambios locales al volver a ejecutar el notebook en el mismo kernel.
reload(mop_client)
reload(release_config)
reload(pipeline)
MOPClient = mop_client.MOPClient
run_pipeline = pipeline.run_pipeline
get_data_release = release_config.get_data_release

DATA_RELEASE_NAME = "DP2"  # DP0.1, DP0.2, DP1 o DP2
DATA_RELEASE = get_data_release(DATA_RELEASE_NAME)

START_DATE = "2026-08-01"
END_DATE = "2026-08-15"
OUTPUT_DIR = Path("outputs")
OBSERVATORY = "El Leoncito"

butler_options = {}
if DATA_RELEASE.butler_collections is not None:
    butler_options["collections"] = DATA_RELEASE.butler_collections
butler = Butler(DATA_RELEASE.butler_repo, **butler_options)
discovery = RSPDiscovery(DATA_RELEASE.rsp_instance)
tap_service = discovery.get_tap_client()
mop = MOPClient()

if not hasattr(mop, "microlensing_parameters"):
    raise RuntimeError(
        "La instalación de mop_api no incluye parámetros tabulares. "
        "Actualizá mop_api desde la rama publicada antes de continuar."
    )


## 2. Plot individual

El reporte muestra coadds, métricas de la colección elegida y fotometría MOP. Usa `OGLE_I` y `G` cuando están disponibles; si faltan ambos, selecciona el filtro con más puntos. La fotometría queda guardada en `outputs/photometry`.


In [9]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates, astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import ImageNormalize, AsinhStretch
from astropy.visualization.wcsaxes import SphericalCircle
from lsst.daf.butler import EmptyQueryResultError
import re

PHOTOMETRY_CACHE_DIR = OUTPUT_DIR / "photometry"
PHOTOMETRY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_event_photometry(target, refresh=False):
    is_row = hasattr(target, "get")
    target_name = target.get("Target") if is_row else str(target)
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(target_name)).strip("._")
    path = PHOTOMETRY_CACHE_DIR / f"{safe_name}.csv"
    if path.exists() and not refresh:
        return pd.read_csv(path, parse_dates=["Timestamp"])
    try:
        return mop.photometry(target_name, save_path=path)
    except Exception as name_error:
        if is_row and target.get("RA") is not None and target.get("Dec") is not None:
            try:
                return mop.photometry(ra=target["RA"], dec=target["Dec"], save_path=path)
            except Exception as position_error:
                name_error = position_error
        result = pd.DataFrame()
        result.attrs["error"] = str(name_error)
        return result

def select_lightcurve_filters(photometry):
    if photometry.empty or "Filter" not in photometry:
        return []
    labels = {str(value).casefold(): value for value in photometry["Filter"].dropna().unique()}
    selected = [labels[key] for key in ("ogle_i", "g") if key in labels]
    if selected:
        return selected
    counts = photometry["Filter"].value_counts()
    return [counts.index[0]] if not counts.empty else []

def prepare_lightcurve_data(photometry, filter_name):
    data = photometry.loc[photometry["Filter"] == filter_name].copy()
    data["Timestamp"] = pd.to_datetime(data["Timestamp"], errors="coerce", utc=True).dt.tz_convert(None)
    data["Magnitude"] = pd.to_numeric(data["Magnitude"], errors="coerce")
    if "Error" in data:
        data["Error"] = pd.to_numeric(data["Error"], errors="coerce")
    return data.dropna(subset=["Timestamp", "Magnitude"]).sort_values("Timestamp")

def plot_target(target, calexps=None, photometry=None, bands="ugrizy", search_radius=11/60, zoom_arcsec=20, circle_arcsec=3, data_release=DATA_RELEASE):
    name, ra, dec = ((target["Target"], target["RA_deg"], target["Dec_deg"])
                     if hasattr(target, "index") else target)
    coord = SkyCoord(ra*u.deg, dec*u.deg)

    # El pipeline entrega estas filas; la consulta queda sólo como fallback.
    if calexps is None:
        query = f"""SELECT {data_release.visit_select("vd")}
                    FROM {data_release.tap_visit_table} AS vd
                    WHERE CONTAINS(POINT('ICRS',vd.{data_release.tap_ra},vd.{data_release.tap_dec}),
                    CIRCLE('ICRS',{ra},{dec},{search_radius}))=1"""
        job = tap_service.submit_job(query); job.run(); job.wait(phases=["COMPLETED","ERROR"])
        if job.phase == "ERROR": job.raise_if_error()
        calexps = job.fetch_result().to_table().to_pandas()

    # Una única consulta Butler devuelve todas las bandas que cubren el punto.
    refs = []
    for dataset_type in data_release.coadd_dataset_types:
        try:
            refs = list(butler.query_datasets(
                dataset_type,
                where=data_release.coadd_spatial_where,
                bind={"ra": ra, "dec": dec},
            ))
        except (EmptyQueryResultError, LookupError):
            refs = []
        if refs:
            break
    coadds = {}
    for ref in refs:
        band = str(ref.dataId["band"])
        if band not in bands or band in coadds:
            continue
        coadd = butler.get(ref)
        x, y = coadd.fits_wcs.world_to_pixel(coord)
        ny, nx = coadd.image.array.shape
        if 0 <= x < nx and 0 <= y < ny:
            coadds[band] = (coadd, x, y)
    if not coadds:
        return None

    fig = plt.figure(figsize=(26,15))
    gs = fig.add_gridspec(5,len(bands),height_ratios=[3,3,1.15,2.1,2.2],hspace=.62,wspace=.55)

    for j, band in enumerate(bands):
        result = coadds.get(band)
        if result is None:
            for row in range(4):
                ax = fig.add_subplot(gs[row,j]); ax.axis("off")
                if row == 0: ax.text(.5,.5,f"Sin coadd {band}",ha="center")
            continue
        coadd, x, y = result

        arr, wcs = np.asarray(coadd.image.array,float), coadd.fits_wcs
        finite = arr[np.isfinite(arr)]; vmin,vmax = np.percentile(finite,[5,99.8])
        norm = ImageNormalize(vmin=vmin,vmax=vmax,stretch=AsinhStretch(.08),clip=True)
        pixscale = np.mean([s.to_value(u.arcsec) for s in wcs.proj_plane_pixel_scales()])
        zoom = zoom_arcsec/pixscale

        for row,lim,title in [(0,None,f"{band}: coadd"),(1,zoom,f"zoom ±{zoom_arcsec}″")]:
            ax = fig.add_subplot(gs[row,j],projection=wcs)
            im = ax.imshow(arr,origin="lower",cmap="gray",norm=norm)
            ax.add_patch(SphericalCircle(coord,circle_arcsec*u.arcsec,transform=ax.get_transform("icrs"),
                                         edgecolor="red",facecolor="none",lw=1.5))
            if lim: ax.set(xlim=(x-lim,x+lim),ylim=(y-lim,y+lim))
            ax.set_title(title,fontsize=10,pad=5)
            ax.coords[0].set_axislabel("RA [deg]",minpad=1.2); ax.coords[1].set_axislabel("Dec [deg]",minpad=.5)
            ax.coords[0].set_major_formatter("d.ddd"); ax.coords[1].set_major_formatter("d.ddd")
            ax.coords[0].set_ticks_position("b")
            ax.coords[0].set_ticklabel_position("b")
            ax.coords[0].set_axislabel_position("b")
            ax.coords[0].set_ticks(direction="out")
            ax.coords[0].set_ticklabel(size=7, rotation=30, pad=10)
            ax.coords[1].set_ticklabel(size=7)
            if row == 0:
                cb = fig.colorbar(im,ax=ax,fraction=.04,pad=.075)
                cb.ax.tick_params(labelsize=7); cb.set_label(str(coadd.unit),fontsize=8)

        d = calexps[calexps.band == band].sort_values("expMidptMJD")
        ax = fig.add_subplot(gs[2,j])
        ax.scatter(d.expMidptMJD,d.seeing,s=12)
        if d.seeing.notna().any(): ax.axhline(d.seeing.median(),ls="--",lw=1,label=f"med={d.seeing.median():.2f}")
        ax.set(xlabel="MJD",ylabel="seeing",title="MJD vs seeing"); ax.grid(alpha=.3)
        ax.tick_params(labelsize=8); ax.legend(fontsize=7,frameon=False)

        n = d[["visitId","detector"]].drop_duplicates().shape[0]
        desc = d[["seeing","magLim"]].describe().round(2)
        cells = [[idx,*values] for idx,values in zip(desc.index,desc.values)]
        ax = fig.add_subplot(gs[3,j]); ax.axis("off")
        tab = ax.table(cellText=cells,colLabels=["","seeing","magLim"],cellLoc="center",
                       colLoc="center",loc="upper center",colWidths=[.20,.29,.29])
        tab.auto_set_font_size(False); tab.set_fontsize(8); tab.scale(1,1.05)
        for cell in tab.get_celld().values(): cell.PAD=.025
        ax.set_title(f"N visitId + detector = {n}",fontsize=9,pad=1)

    ax_lc = fig.add_subplot(gs[4, :])
    photometry = load_event_photometry(name) if photometry is None else photometry
    selected_filters = select_lightcurve_filters(photometry)
    if not selected_filters:
        message = photometry.attrs.get("error", "Sin fotometría disponible")
        ax_lc.text(.5, .5, message, ha="center", va="center", transform=ax_lc.transAxes)
        ax_lc.set_axis_off()
    else:
        for filter_name in selected_filters:
            data = prepare_lightcurve_data(photometry, filter_name)
            if data.empty:
                continue
            dates = mdates.date2num(data["Timestamp"].to_numpy(dtype="datetime64[us]"))
            magnitudes = data["Magnitude"].to_numpy(dtype=float)
            errors = data["Error"].to_numpy(dtype=float) if "Error" in data else None
            valid = np.isfinite(dates) & np.isfinite(magnitudes)
            if errors is not None:
                errors = np.where(np.isfinite(errors[valid]) & (errors[valid] >= 0), errors[valid], np.nan)
            ax_lc.errorbar(dates[valid], magnitudes[valid], yerr=errors, fmt=".", ms=3, alpha=.75, label=f"{filter_name} (N={valid.sum()})")
        locator = mdates.AutoDateLocator(minticks=4, maxticks=10)
        ax_lc.xaxis.set_major_locator(locator)
        ax_lc.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax_lc.invert_yaxis()
        ax_lc.set(xlabel="Fecha", ylabel="Magnitud", title="Fotometría MOP")
        ax_lc.grid(alpha=.25)
        ax_lc.legend(frameon=False, ncol=len(selected_filters))

    priority = any(str(target.get(k, "")).strip().lower() not in {"","0","0.0","false","nan","none"}
                   for k in ("priority","tap_priority","tap_priority_longte") if hasattr(target, "get"))
    flag = " [PRIORITY]" if priority else ""
    meta_cols = [c for c in ("mag_now","Min airmass","n_visible_nights","coverage_n_visits") if c in target.index]
    mop_cols = [c for c in target.index if str(c).startswith("mop_") and c not in {"mop_link", "mop_parameters_error"}]
    preferred = ("t_e", "te", "u_0", "u0", "t_0", "t0", "rho", "pi_e", "magnitude", "baseline", "parameters_status")
    mop_cols.sort(key=lambda c: (not any(key in str(c).lower() for key in preferred), str(c)))

    label_names = {
        "mag_now": "Magnitud actual", "Min airmass": "Airmass mínimo",
        "n_visible_nights": "Noches visibles", "coverage_n_visits": "Visitas Rubin",
    }
    def panel_line(column):
        raw_label = str(column).removeprefix("mop_")
        label = label_names.get(column, raw_label)
        value = target[column]
        if isinstance(value, (float, np.floating)):
            value = f"{value:.4g}"
        unit = " days" if raw_label.casefold() in {"t_e", "te"} and "day" not in str(value).casefold() else ""
        return f"{label}: {value}{unit}"

    valid_cols = [c for c in meta_cols + mop_cols if str(target[c]).strip().lower() not in {"nan", "none", ""}]
    panel_lines = ["DATOS DEL TARGET", "", f"RA: {ra:.5f}°", f"Dec: {dec:.5f}°", f"Filas cobertura: {len(calexps)}"]
    if priority:
        panel_lines.extend(["", "PRIORITY"])
    if valid_cols:
        panel_lines.extend(["", "PARÁMETROS", ""] + [panel_line(c) for c in valid_cols])

    fig.suptitle(f"{name}{flag}", y=.985, fontsize=14,
                 color="crimson" if priority else "black",
                 fontweight="bold" if priority else "normal")
    fig.text(.825, .91, "\n".join(panel_lines), ha="left", va="top", fontsize=9.5,
             linespacing=1.45, family="monospace",
             bbox=dict(boxstyle="round,pad=.7", facecolor="whitesmoke", edgecolor="0.75"))
    fig.subplots_adjust(top=.94,bottom=.055,left=.04,right=.79)
    return fig

## 3. Ejecutar el pipeline

Cada página MOP se descarga una sola vez con concurrencia limitada y produce parámetros más fotometría. Los CSV, páginas MOP, 404, fotometrías, consultas de cobertura Rubin y plots quedan cacheados; la celda informa el progreso por etapas. Usá `reuse_cache=False` sólo para forzar datos nuevos.


In [10]:
combined, paths = run_pipeline(
    mop=mop,
    tap_service=tap_service,
    start_date=START_DATE,
    end_date=END_DATE,
    root_dir=OUTPUT_DIR,
    observatory=OBSERVATORY,
    target_plotter=lambda row, coverage: plot_target(
        row, calexps=coverage, photometry=load_event_photometry(row)
    ),
    max_workers=4,
    reuse_cache=True,
    overwrite_target_plots=False,
    verbose=True,
    data_release=DATA_RELEASE,
)

print(f"Targets procesados: {len(combined)}")
print(f"Productos guardados en: {paths['run'].resolve()}")
combined.head()


[1/5] Targets visibles MOP (cache)
[2/5] Parámetros + fotometría MOP (cache)
[3/5] Cobertura DP2 (cache)
[4/5] Tablas y mapas
[5/5] Reportes por target
    targets revisados: 1/53 | generados=0, migrados=1, cache=0, sin coadd=0, errores=0
    targets revisados: 10/53 | generados=0, migrados=6, cache=5, sin coadd=4, errores=0
    targets revisados: 20/53 | generados=0, migrados=15, cache=14, sin coadd=5, errores=0
    targets revisados: 30/53 | generados=0, migrados=19, cache=19, sin coadd=10, errores=0
    targets revisados: 40/53 | generados=0, migrados=23, cache=21, sin coadd=18, errores=0
    targets revisados: 50/53 | generados=0, migrados=25, cache=23, sin coadd=26, errores=0
    targets revisados: 53/53 | generados=0, migrados=26, cache=24, sin coadd=28, errores=0
    reportes terminados: generados=0, migrados=26, cache=24, sin coadd=29, errores=0
Pipeline completo en 0.1 min
Targets procesados: 53
Productos guardados en: /home/karennowo/target_selection/outputs/2026-08-01_to_202

,Target,RA,Dec,Site,Min airmass,mag_now,tap_priority,tap_priority_longte,mop_link,observation_date,...,coverage_n_calexps,coverage_first_mjd,coverage_last_mjd,coverage_bands,coverage_n_visits_g,coverage_n_visits_r,coverage_n_visits_i,coverage_n_visits_z,coverage_n_visits_y,coverage_n_visits_u
0,OGLE-2024-BLG-0808,18h24m19.2s,-17d04m27.5s,(El Leoncito) El Leoncito,1.0,16.580734,0.77774,54.49391,https://mop.lco.global/targets/45864/,2026-08-01,...,87.0,60848.107026,60880.217058,"g,i,r,u,y,z",5.0,5.0,10.0,9.0,11.0,6.0
1,OGLE-2026-BLG-0239,18h33m08.86s,-20d11m14.6s,(El Leoncito) El Leoncito,1.0,16.836891,0.74951,9.79946,https://mop.lco.global/targets/56180/,2026-08-01,...,75.0,60847.341350,60880.216097,"g,i,r,u,y,z",9.0,8.0,6.0,6.0,6.0,6.0
2,OGLE-2026-GD-0001,16h28m35.32s,-45d16m14.5s,(El Leoncito) El Leoncito,1.0,16.978988,2.17173,23.55821,https://mop.lco.global/targets/56074/,2026-08-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OGLE-2025-BLG-0271,17h54m29.54s,-39d44m25.3s,(El Leoncito) El Leoncito,1.0,17.005745,1.08405,73.50817,https://mop.lco.global/targets/53412/,2026-08-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OGLE-2025-BLG-0242,17h13m08.02s,-32d15m43.3s,(El Leoncito) El Leoncito,1.0,17.131037,2.13822,89.43000,https://mop.lco.global/targets/53379/,2026-08-01,...,69.0,60851.051098,60880.230235,"g,i,r,y,z",4.0,9.0,10.0,13.0,2.0,0.0


## 4. Verificar los productos

Comprueba que los parámetros `mop_*` estén tanto en los CSV como en el DataFrame que recibe `plot_target`, y lista los reportes realmente generados.


In [11]:
summary_path = paths["tables"] / "visible_summary.csv"
combined_path = paths["tables"] / "combined_targets.csv"
summary_csv = pd.read_csv(summary_path)
combined_csv = pd.read_csv(combined_path)

parameter_columns = [
    column for column in combined.columns
    if column.startswith("mop_")
    and column not in {"mop_link", "mop_parameters_status", "mop_parameters_error"}
]
status_counts = (
    combined["mop_parameters_status"].value_counts(dropna=False).to_dict()
    if "mop_parameters_status" in combined else {}
)

missing_summary = sorted(set(parameter_columns) - set(summary_csv.columns))
missing_combined = sorted(set(parameter_columns) - set(combined_csv.columns))
if missing_summary or missing_combined:
    raise RuntimeError(
        f"Parámetros ausentes en CSV: summary={missing_summary}, "
        f"combined={missing_combined}"
    )

report_paths = sorted(paths["targets"].glob("*_target_report.png"))
print(f"Estado de páginas MOP: {status_counts}")
print(f"Columnas de parámetros medidos ({len(parameter_columns)}):")
print(parameter_columns or "Ninguna página accesible devolvió parámetros medidos")
print(f"Reportes generados para targets con coadd: {len(report_paths)}")
print(f"CSV summary: {summary_path}")
print(f"CSV combinado: {combined_path}")
combined[["Target", *parameter_columns]].head()

summary_csv = paths["tables"] / "target_summary.csv"
summary_png = paths["tables"] / "target_summary.png"
print(f"Tabla resumen: {summary_csv.resolve()}")
print(f"PNG resumen: {summary_png.resolve()}")


Estado de páginas MOP: {'available': 36, 'unavailable': 17}
Columnas de parámetros medidos (13):
['mop_link_mop', 'mop_alive', 'mop_observing_mode', 'mop_tap_priority', 'mop_tap_priority_longte', 'mop_interferometry_candidate', 'mop_t_0_hjd', 'mop_u_0', 'mop_t_e_days', 'mop_mag_now', 'mop_latest_data_hjd', 'mop_latest_data_utc', 'mop_last_fit_jd']
Reportes generados para targets con coadd: 24
CSV summary: outputs/2026-08-01_to_2026-08-15/tables/visible_summary.csv
CSV combinado: outputs/2026-08-01_to_2026-08-15/tables/combined_targets.csv
Tabla resumen: /home/karennowo/target_selection/outputs/2026-08-01_to_2026-08-15/tables/target_summary.csv
PNG resumen: /home/karennowo/target_selection/outputs/2026-08-01_to_2026-08-15/tables/target_summary.png
